# ML 기반
- k-NN
    - 숫자데이터로 유클리디언 거리 계산 기반
    - 모두 정규화 후 k = 5 했을 때 성공확률이 높은 상위 3개 보여주기, 고객 예산에 맞는 것까지 계산하여 (월세/보증금 등)

In [ ]:
# k-NN 예시 코드 (gpt)
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. 예시 데이터: (보증금, 월세, 면적, 유동인구, 경쟁 카페 수)
X = np.array([
    [5000, 200, 30, 500, 10],    # 매물 0
    [10000, 300, 45, 700, 5],    # 매물 1
    [3000, 150, 28, 400, 12],    # 매물 2
    [8000, 250, 35, 650, 7],     # 매물 3
    [6000, 220, 32, 520, 8],     # 매물 4
    [4000, 180, 29, 480, 9]      # 매물 5
])

# 2. 성공 여부 (라벨, 1=성공, 0=실패)
y = np.array([1, 1, 0, 1, 0, 1])

# 3. 데이터 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. k-NN 학습 (k=5)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_scaled, y)

# 5. 고객 입력 정보
customer_budget_deposit = 7000   # 고객 보증금 한도
customer_budget_rent = 250       # 고객 월세 한도

# 6. 고객 예산에 맞는 매물 필터링
candidate_idx = (X[:, 0] <= customer_budget_deposit) & (X[:, 1] <= customer_budget_rent)
X_candidate = X_scaled[candidate_idx]

# 매물 원본 인덱스도 저장
candidate_original_idx = np.where(candidate_idx)[0]

# 7. 필터링된 매물들에 대해 성공 확률 예측
success_prob = knn.predict_proba(X_candidate)[:, 1]  # 성공(1) 확률만 가져옴

# 8. 성공 확률 높은 순으로 정렬
recommended_idx = np.argsort(success_prob)[::-1]  # 높은 확률 순

# 9. 상위 3개 추천
top_k = 3
top_k_recommendations = candidate_original_idx[recommended_idx[:top_k]]
top_k_probs = success_prob[recommended_idx[:top_k]]

# 10. 결과 출력
print("✅ 고객 예산에 맞는 성공 확률 높은 추천 매물 Top 3")
for i, idx in enumerate(top_k_recommendations):
    print(f"{i+1}. 매물 번호 {idx} (성공 확률: {top_k_probs[i]:.2f})")


# Clustering
KMeans, DBSCAN, Hierarchical Clustering
➔ feature 벡터로 클러스터링 → "성공 카페 클러스터"와 거리 계산

- ✅ KMeans: 대표적, 빠르고 간단
- Clustering은 따로 라벨링 없이 군집화 : 실제 우리 기준과 다를 수 있음



# 유사도 기반
Cosine Similarity or Euclidean Distance로 유사도 계산
➔ 성공한 카페들과 유사도 평균을 내서 추천

- ✅ Cosine: 방향(패턴)이 중요할 때 유리
- cosine similarity 구하는 방법 (정규화?)
- 성공 카페의 평균 벡터를 만들고 이와 가장 유사한 매물을 구한다 (해당 상권 내 or 전체)

In [ ]:
# cosine_similarity 기반 에시코드 (gpt)
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

# 1. 예시 데이터 (보증금, 월세, 면적, 유동인구, 경쟁 카페 수)
X = np.array([
    [5000, 200, 30, 500, 10],    # 매물 0
    [10000, 300, 45, 700, 5],    # 매물 1
    [3000, 150, 28, 400, 12],    # 매물 2
    [8000, 250, 35, 650, 7],     # 매물 3
    [6000, 220, 32, 520, 8],     # 매물 4
    [4000, 180, 29, 480, 9]      # 매물 5
])

# 2. 성공 여부 (1=성공, 0=실패)
y = np.array([1, 1, 0, 1, 0, 1])

# 3. 스케일링 (표준화)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. 성공한 매물들만 뽑기
success_X = X_scaled[y == 1]

# 5. 성공 매물들의 평균 패턴 벡터 만들기
success_mean_vector = np.mean(success_X, axis=0).reshape(1, -1)

# 6. 모든 매물과 성공 평균 패턴 간 cosine 유사도 계산
similarities = cosine_similarity(X_scaled, success_mean_vector).flatten()

# 7. 고객 예산 조건
customer_budget_deposit = 7000   # 보증금 한도
customer_budget_rent = 250       # 월세 한도

# 8. 예산 조건에 맞는 매물만 필터링
budget_mask = (X[:, 0] <= customer_budget_deposit) & (X[:, 1] <= customer_budget_rent)
candidate_idx = np.where(budget_mask)[0]

# 9. 후보 매물들의 유사도만 가져오기
candidate_similarities = similarities[candidate_idx]

# 10. 유사도 높은 순 정렬
sorted_idx = np.argsort(candidate_similarities)[::-1]

# 11. 상위 3개 추천
top_k = 3
top_candidates = candidate_idx[sorted_idx[:top_k]]
top_scores = candidate_similarities[sorted_idx[:top_k]]

# 12. 결과 출력
print("✅ 고객 예산에 맞는 성공 패턴 유사 추천 매물 Top 3")
for i, idx in enumerate(top_candidates):
    print(f"{i+1}. 매물 번호 {idx} (성공 패턴 유사도: {top_scores[i]:.2f})")


✅ 고객 예산에 맞는 성공 패턴 유사 추천 매물 Top 3
1. 매물 번호 4 (성공 패턴 유사도: -0.15)
2. 매물 번호 5 (성공 패턴 유사도: -0.93)
3. 매물 번호 0 (성공 패턴 유사도: -0.96)


# 딥러닝 기반 Representation Learning

- AutoEncoder → feature를 압축해서 latent space로 매핑하고 유사도 계산

- Siamese Network → 유사성 학습을 직접 시켜서, 비슷한 매물 판단

- ✅ 데이터 양 많으면 AutoEncoder나 Siamese로 더 정교한 추천 가능
    - 학습 dataset 약 13000~15000개로 추정
    - 매물 개수 다시 확인할 필요
    - 딥러닝이 필요할까?